# 국가핵심기술 키워드 ↔ 표준 용어 유사도 분석

**목표**: 국가핵심기술 키워드와 의미가 비슷한 표준 용어(`df_word`)를 임베딩 유사도로 찾는다.

**진행 순서**
1. 설정 및 데이터 불러오기
2. 키워드 리스트 만들기 (`L5` + `keyword` 합치기 → `,` 기준으로 쪼개기)
3. 사내 LLM API로 임베딩 생성
4. 코사인 유사도 계산
5. 결과 확인 및 저장

## 1. 설정 및 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [ ]:
import os

# 사내 LLM API 설정 (주소와 키는 환경변수로 설정)
BASE_URL = os.environ.get("LLM_BASE_URL", "http://<사내 LLM 주소>/v1")
EMBEDDING_URL = f"{BASE_URL}/embeddings"

OPEN_API_KEY = os.environ.get("LLM_API_KEY", "<사내 API 키>")   # 사내 embeddings API 구독 필요
MODEL_NAME = "bge-m3"

HEADERS = {
    "Authorization": f"Bearer {OPEN_API_KEY}",
    "Content-Type": "application/json",
}

In [ ]:
# 데이터 불러오기 (파일 경로/형식은 실제에 맞게 수정)
df_keyword = pd.read_excel("국가핵심기술_키워드.xlsx")   # 컬럼: L5, keyword
df_word = pd.read_excel("표준용어.xlsx")                 # 컬럼: 설명 포함

print("df_keyword:", df_keyword.shape)
print("df_word   :", df_word.shape)

In [ ]:
df_keyword.head()

In [ ]:
df_word.head()

## 2. 키워드 리스트 만들기

- `L5` 열과 `keyword` 열의 값을 하나의 열로 합친다.
- 한 칸에 `,`로 여러 값이 들어 있으면 각각 쪼갠다.
- 앞뒤 공백 제거, 빈 값·중복 제거
- 나중에 추적할 수 있도록 **어느 L5에서 나온 키워드인지**도 함께 남겨둔다.

In [ ]:
# L5 값 자체도 키워드로 사용 → (L5, 값) 형태로 세로로 쌓기
df_from_l5 = df_keyword[["L5"]].copy()
df_from_l5["keyword_raw"] = df_keyword["L5"]

df_from_kw = df_keyword[["L5"]].copy()
df_from_kw["keyword_raw"] = df_keyword["keyword"]

df_kw_long = pd.concat([df_from_l5, df_from_kw], ignore_index=True)
df_kw_long = df_kw_long.dropna(subset=["keyword_raw"])

print("합친 후 행 수:", len(df_kw_long))
df_kw_long.head()

In [ ]:
# ',' 기준으로 쪼개서 한 행에 하나의 키워드만 남기기
df_kw_long["keyword"] = df_kw_long["keyword_raw"].astype(str).str.split(",")
df_kw_long = df_kw_long.explode("keyword")

# 공백 제거 후 빈 문자열 제거
df_kw_long["keyword"] = df_kw_long["keyword"].str.strip()
df_kw_long = df_kw_long[df_kw_long["keyword"] != ""]

df_kw_long = df_kw_long[["L5", "keyword"]].drop_duplicates().reset_index(drop=True)

print("쪼갠 후 행 수:", len(df_kw_long))
df_kw_long.head(20)

In [ ]:
# 임베딩할 고유 키워드 리스트 (같은 키워드가 여러 L5에 걸쳐 있을 수 있으므로 중복 제거)
keyword_list = df_kw_long["keyword"].drop_duplicates().tolist()

print("고유 키워드 수:", len(keyword_list))
keyword_list[:20]

## 3. 임베딩 생성

- 한 번에 여러 문장을 보내 호출 횟수를 줄인다. (`BATCH_SIZE`개씩)
- API 호출은 시간이 걸리므로 결과를 `.npy` 파일로 저장해 두고 재사용한다.

In [ ]:
def get_embeddings(texts, batch_size=32):
    """texts 리스트를 batch_size개씩 나눠 임베딩하고 (문장 수, 차원) 배열로 반환"""
    all_vectors = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        body = {"input": batch, "model": MODEL_NAME}

        rsp = requests.post(EMBEDDING_URL, headers=HEADERS, json=body, timeout=60)
        rsp.raise_for_status()   # 오류 발생 시 바로 멈춤
        rst = rsp.json()

        # 응답 순서를 index 기준으로 정렬해서 입력 순서와 맞춤
        data = sorted(rst["data"], key=lambda x: x["index"])
        all_vectors.extend([d["embedding"] for d in data])

    return np.array(all_vectors)

In [ ]:
# API 연결 테스트
test_vec = get_embeddings(["hello"])
print("임베딩 차원:", test_vec.shape)

In [ ]:
# 키워드 임베딩
keyword_emb = get_embeddings(keyword_list)
np.save("keyword_emb.npy", keyword_emb)

print(keyword_emb.shape)

In [ ]:
# 표준 용어 '설명' 임베딩 (설명이 비어 있는 행은 제외)
df_word_valid = df_word[df_word["설명"].notna()].copy()
df_word_valid["설명"] = df_word_valid["설명"].astype(str).str.strip()
df_word_valid = df_word_valid[df_word_valid["설명"] != ""].reset_index(drop=True)

print("전체 표준 용어 수   :", len(df_word))
print("설명 있는 용어 수   :", len(df_word_valid))

desc_list = df_word_valid["설명"].tolist()
desc_emb = get_embeddings(desc_list)
np.save("desc_emb.npy", desc_emb)

print(desc_emb.shape)

In [ ]:
# (재실행 시) API를 다시 호출하지 않고 저장된 임베딩 불러오기
# keyword_emb = np.load("keyword_emb.npy")
# desc_emb = np.load("desc_emb.npy")

## 4. 코사인 유사도 계산

벡터 길이를 1로 맞춘(정규화) 뒤 행렬곱을 하면 코사인 유사도가 된다.
결과 `sim`의 모양은 **(키워드 수, 표준 용어 수)**.

In [ ]:
def normalize(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

sim = normalize(keyword_emb) @ normalize(desc_emb).T

print("유사도 행렬 shape:", sim.shape)
print(f"유사도 범위: {sim.min():.3f} ~ {sim.max():.3f}")

In [ ]:
# 유사도 분포 확인 → 임계값(THRESHOLD) 정할 때 참고
pd.Series(sim.flatten()).describe(percentiles=[0.5, 0.9, 0.99, 0.999])

## 5. 결과 확인

### 5-1. 키워드별 유사도 상위 N개 표준 용어

In [ ]:
TOP_N = 5

rows = []
for i, kw in enumerate(keyword_list):
    top_idx = np.argsort(-sim[i])[:TOP_N]      # 유사도 높은 순으로 인덱스 정렬
    for rank, j in enumerate(top_idx, start=1):
        rows.append({
            "keyword": kw,
            "rank": rank,
            "similarity": round(float(sim[i, j]), 4),
            "word_idx": j,
        })

df_top = pd.DataFrame(rows)

# 표준 용어 정보 붙이기
df_top = df_top.merge(df_word_valid, left_on="word_idx", right_index=True, how="left")

# 어느 L5의 키워드인지 붙이기 (한 키워드가 여러 L5에 속하면 ', '로 묶어서 표시)
kw_to_l5 = df_kw_long.groupby("keyword")["L5"].apply(lambda x: ", ".join(x.astype(str).unique()))
df_top.insert(0, "L5", df_top["keyword"].map(kw_to_l5))

df_top.head(30)

### 5-2. 임계값 이상인 표준 용어만 보기

- 아래 `THRESHOLD`는 임시 값. 위 유사도 분포와 실제 결과를 보면서 조정한다.
- 한 표준 용어가 여러 키워드와 매칭될 수 있으므로, **용어 기준으로 가장 유사한 키워드**를 남긴다.

In [ ]:
THRESHOLD = 0.6

# 표준 용어별로 가장 유사한 키워드와 그 점수
best_kw_idx = sim.argmax(axis=0)      # 각 표준 용어(열)에서 최대값이 나온 키워드(행) 번호
best_score = sim.max(axis=0)

df_result = df_word_valid.copy()
df_result["best_keyword"] = [keyword_list[k] for k in best_kw_idx]
df_result["best_L5"] = df_result["best_keyword"].map(kw_to_l5)
df_result["similarity"] = best_score.round(4)

df_matched = (
    df_result[df_result["similarity"] >= THRESHOLD]
    .sort_values("similarity", ascending=False)
    .reset_index(drop=True)
)

print(f"임계값 {THRESHOLD} 이상 표준 용어 수: {len(df_matched)} / {len(df_result)}")
df_matched.head(50)

In [ ]:
# L5별로 몇 개 매칭됐는지 확인
df_matched["best_L5"].value_counts()

### 5-3. 결과 저장

In [ ]:
with pd.ExcelWriter("국가핵심기술_표준용어_매칭결과.xlsx") as writer:
    df_matched.to_excel(writer, sheet_name="임계값이상_매칭", index=False)
    df_top.to_excel(writer, sheet_name=f"키워드별_Top{TOP_N}", index=False)
    df_kw_long.to_excel(writer, sheet_name="키워드리스트", index=False)

print("저장 완료")